# Module 3: Content Understanding - AI-Powered Document Analysis

## 🔄 Building on Module 2

In **Module 2**, we solved the structure problem with Document Intelligence:
- ✅ Tables → Preserved rows/columns
- ✅ Figures → Detected with bounding boxes
- ✅ Paragraphs → Identified roles (header, footer, content)

**But there's still a gap:**

| What DI Gives Us | What RAG Needs |
|------------------|----------------|
| Figure at `[x, y, width, height]` | **Description**: "A zoning map showing residential areas in pink..." |
| Chart bounding box | **Data**: The actual values and trends |
| Diagram location | **Explanation**: What the diagram represents |

**Content Understanding bridges this gap** using GPT-4.1-mini to generate semantic descriptions automatically.

## 🎯 Learning Objectives

By the end of this module, you will:
1. Understand what **Content Understanding** adds beyond Document Intelligence
2. Use the `prebuilt-documentSearch` analyzer for RAG-optimized extraction
3. See how figures get **AI-generated descriptions** (the "invisible" becomes "describable")
4. Learn **Semantic Chunking** using markdown headers

---

## Step 0: Let's See What We're Working With

Before we start, let's look at **Page 1 of `metro-s36.pdf`** - the same document from Modules 1 and 2.

This page has:
- 🗺️ A **zoning map** (800m radius around Station 36)
- 📷 Three **street photos** showing different views
- 📊 A **legend** with color-coded land use categories

**In Module 1**: These were completely invisible to our RAG pipeline.  
**In Module 2**: We detected their locations (bounding boxes).  
**In Module 3**: We'll get **AI descriptions** of what's IN them!

In [ ]:
# First, let's render Page 1 of our PDF so you can see what we're analyzing
# This requires pdf2image (optional - will show placeholder if not available)

from pathlib import Path
from IPython.display import display, Image, Markdown, HTML
import os

DATA_DIR = Path("../../data/sample-pdfs")
PDF_PATH = DATA_DIR / "metro-s36.pdf"

print(f"📄 Target Document: {PDF_PATH.name}")
print(f"   This is the SAME document from Modules 1 and 2!\n")

# Try to render the first page as an image
try:
    from pdf2image import convert_from_path
    
    print("🖼️ Rendering Page 1 for reference...")
    pages = convert_from_path(PDF_PATH, first_page=1, last_page=1, dpi=150)
    
    # Save temporarily for display
    page1_path = "page1_preview.png"
    pages[0].save(page1_path, "PNG")
    
    print("\n👀 HERE'S WHAT WE'RE ANALYZING (Page 1):")
    print("="*60)
    display(Image(filename=page1_path, width=600))
    print("="*60)
    print("\n🎯 Notice:")
    print("   - The MAP with colored zones (מגורים, תעסוקה, מסחר, etc.)")
    print("   - The PHOTOS showing street views")
    print("   - The LEGEND explaining the color codes")
    print("\n   Content Understanding will DESCRIBE all of these!")
    
except ImportError:
    print("⚠️ pdf2image not installed. Showing text description instead.")
    print("\n📄 Page 1 of metro-s36.pdf contains:")
    print("   ┌─────────────────────────────────────────────────────┐")
    print("   │  🗺️ ZONING MAP           │  📋 STATION INFO        │")
    print("   │  (800m radius)           │  - מיקום התחנה          │")
    print("   │  Colors show land use:   │  - סוג תחנה             │")
    print("   │  - Pink = Residential    │  - קיבולת נוסעים        │")
    print("   │  - Blue = Commercial     │                         │")
    print("   │                          │                         │")
    print("   ├──────────┬──────────┬────┴─────────────────────────┤")
    print("   │  📷 East │  📷 South│  📷 North                    │")
    print("   │  View    │  View    │  View                        │")
    print("   └──────────┴──────────┴──────────────────────────────┘")
    print("\n   💡 To see the actual PDF, open: data/sample-pdfs/metro-s36.pdf")

---

## Step 1: Setup - Initialize Clients

We need to set up:
1. **Content Understanding Client** - For semantic analysis with `prebuilt-documentSearch`
2. **Credentials** - Using Entra ID (same as Module 2)

### 🔑 Key Difference: Content Understanding vs Document Intelligence

| Aspect | Document Intelligence | Content Understanding |
|--------|----------------------|----------------------|
| **Endpoint** | `*.cognitiveservices.azure.com` | `*.services.ai.azure.com` |
| **Figure Output** | Bounding box only | Bounding box + **AI description** |
| **Requires LLM** | No | Yes (GPT-4.1-mini) |
| **Cost** | Lower | Higher (LLM calls) |

In [ ]:
# Install the Content Understanding SDK if needed
%pip install azure-ai-contentunderstanding -q

In [ ]:
import os
import sys
import json
import re
from pathlib import Path

# Add src to path for shared utilities
sys.path.append(str(Path("../../src").resolve()))

from utils import load_env
from IPython.display import display, Image, Markdown, HTML
from azure.identity import DefaultAzureCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient

# Load environment variables
env = load_env()

# --- Setup Credentials ---
print("🔐 Initializing with DefaultAzureCredential (Entra ID)...")
credential = DefaultAzureCredential()

# --- Setup Content Understanding Client ---
# CU uses a different endpoint format than Document Intelligence
doc_endpoint = env["AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"]
cu_endpoint = doc_endpoint.replace(".cognitiveservices.azure.com", ".services.ai.azure.com")

print(f"\n📍 Endpoints:")
print(f"   Document Intelligence: {doc_endpoint}")
print(f"   Content Understanding: {cu_endpoint}")

# Initialize client with GA API version
API_VERSION = "2025-11-01"  # GA version
cu_client = ContentUnderstandingClient(
    endpoint=cu_endpoint,
    credential=credential,
    api_version=API_VERSION
)

# Define paths
DATA_DIR = Path("../../data/sample-pdfs")
PDF_PATH = DATA_DIR / "metro-s36.pdf"

print(f"\n✅ Content Understanding Client Ready (API: {API_VERSION})")
print(f"   Target document: {PDF_PATH.name}")

---

## Step 2: Configure Model Deployments

Content Understanding's `prebuilt-documentSearch` analyzer uses **GPT-4.1-mini** internally to:
- Generate semantic descriptions for figures
- Convert charts to Chart.js code
- Convert diagrams to Mermaid.js syntax

We need to tell CU which Azure OpenAI deployment to use.

### ⚠️ Important: Model Mapping Required

CU expects specific model names mapped to your Azure OpenAI deployments:

| CU Model Name | Your Deployment Name |
|---------------|---------------------|
| `gpt-4.1-mini` | (from your Azure OpenAI resource) |
| `text-embedding-3-large` | (optional, for embeddings) |

In [ ]:
# Configure Content Understanding to use your Azure OpenAI deployments
# This is a ONE-TIME setup per resource

print("⚙️ Configuring Content Understanding Model Deployments...\n")

# Get deployment names from environment
gpt41_deployment = env.get("GPT_4_1_DEPLOYMENT", "gpt-4.1")
gpt41_mini_deployment = env.get("GPT_4_1_MINI_DEPLOYMENT", "gpt-4.1-mini")
embedding_deployment = env.get("TEXT_EMBEDDING_3_LARGE_DEPLOYMENT", "text-embedding-3-large")

print(f"   📋 Model deployments to configure:")
print(f"      gpt-4.1 → {gpt41_deployment}")
print(f"      gpt-4.1-mini → {gpt41_mini_deployment}  ← Required for prebuilt-documentSearch")
print(f"      text-embedding-3-large → {embedding_deployment}")

try:
    # Check current configuration
    print("\n   🔎 Checking current configuration...")
    current_defaults = cu_client.get_defaults()
    
    if current_defaults.model_deployments:
        print(f"   Current: {current_defaults.model_deployments}")
    else:
        print("   No defaults configured yet.")
    
    # Update with our deployment mapping
    model_deployments = {
        "gpt-4.1": gpt41_deployment,
        "gpt-4.1-mini": gpt41_mini_deployment,
        "text-embedding-3-large": embedding_deployment,
    }
    
    print("\n   🔧 Updating defaults...")
    updated_defaults = cu_client.update_defaults(model_deployments=model_deployments)
    print("   ✅ Defaults configured successfully!")
    
    if updated_defaults.model_deployments:
        print("\n   Final configuration:")
        for model_name, deployment_name in updated_defaults.model_deployments.items():
            print(f"      {model_name}: {deployment_name}")

except Exception as e:
    print(f"\n❌ Configuration failed: {e}")
    print("\n   ⚠️ TROUBLESHOOTING:")
    print("      1. Ensure you have 'Cognitive Services User' role on the resource")
    print("      2. Verify model deployments exist in Azure AI Foundry")
    print("      3. Check that deployment names in .env match your Azure deployments")

---

## Step 3: Analyze Document with Content Understanding

Now for the magic! We'll use the `prebuilt-documentSearch` analyzer which:

1. **Extracts text** with reading order (like DI)
2. **Detects figures** with bounding boxes (like DI)
3. **Generates AI descriptions** for each figure (NEW!)
4. **Outputs GitHub Flavored Markdown** optimized for LLMs

### ⏱️ Timing Note

Content Understanding takes longer than Document Intelligence because it:
- Crops each detected figure
- Sends each figure to GPT-4.1-mini for description
- Generates structured markdown output

**Expect 3-10 minutes** depending on document size and figure count.

### 💡 Caching Strategy

In production, you'd cache CU results to avoid re-processing. We'll save results to a JSON file.

In [ ]:
# ⏱️ This cell analyzes the PDF with Content Understanding
# Expected time: 3-10 minutes (depends on figures and document size)
#
# 💡 TIP: If you want to skip the wait, run the NEXT cell instead
#    which loads pre-cached results.

CACHE_FILE = "metro_s36_cu_result.json"
analyzer_id = "prebuilt-documentSearch"

print(f"🔍 Analyzing {PDF_PATH.name} with Content Understanding...")
print(f"   Analyzer: {analyzer_id}")
print(f"   ⏱️  This may take 3-10 minutes. Please wait...\n")

# Read the PDF
with open(PDF_PATH, "rb") as f:
    file_bytes = f.read()

print(f"   File size: {len(file_bytes):,} bytes ({len(file_bytes)/1024/1024:.1f} MB)")

try:
    # Start analysis
    print("\n   Starting analysis...")
    
    if hasattr(cu_client, "begin_analyze_binary"):
        response = cu_client.begin_analyze_binary(
            analyzer_id=analyzer_id,
            binary_input=file_bytes,
            content_type="application/pdf"
        )
    else:
        response = cu_client.begin_analyze(
            analyzer_id=analyzer_id,
            body=file_bytes,
            content_type="application/pdf"
        )
    
    print("   Waiting for results (the LLM is describing each figure)...")
    result = response.result()
    
    print("\n✅ Analysis Complete!")
    
    # Save to cache
    if hasattr(result, "as_dict"):
        result_dict = result.as_dict()
    else:
        result_dict = {"content": str(result)}
    
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(result_dict, f, indent=2, ensure_ascii=False)
    
    print(f"   💾 Results cached to: {CACHE_FILE}")

except Exception as e:
    print(f"\n❌ Analysis failed: {e}")
    print("\n   ⚠️ Try loading cached results in the next cell instead.")
    result_dict = None

In [ ]:
# ⏩ ALTERNATIVE: Load cached results (skip the 3-10 minute wait)
# Use this if you already ran the analysis before or want to skip waiting

CACHE_FILE = "metro_s36_cu_result.json"

# Initialize markdown_text
markdown_text = ""

if Path(CACHE_FILE).exists():
    print(f"📂 Loading cached results from: {CACHE_FILE}")
    
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        result_dict = json.load(f)
    
    # Extract markdown from the result
    contents_list = result_dict.get("contents") or result_dict.get("result", {}).get("contents", [])
    
    if contents_list:
        markdown_text = contents_list[0].get("markdown", "")
        print(f"\n✅ Loaded {len(markdown_text):,} characters of markdown")
        print(f"   Found {markdown_text.count('!['):,} figure references")
        print(f"   Found {markdown_text.count('#'):,} markdown headers")
    else:
        # Fallback for different result structure
        markdown_text = result_dict.get("content", "")
        print(f"\n✅ Loaded {len(markdown_text):,} characters")
else:
    print(f"❌ Cache file not found: {CACHE_FILE}")
    print("   👉 Run the analysis cell above first.")

---

## Step 4: Inspect the Results - The "Invisible" is Now Visible!

Remember in Module 1, when we asked about the zoning map, our RAG pipeline had **no idea** what was in it?

Now let's see what Content Understanding extracted. The key breakthrough is:

**Figures now have AI-generated descriptions!**

The markdown format is:
```markdown
![ALT_TEXT](figures/PAGE.FIGURE_NUM "AI_SEMANTIC_DESCRIPTION")
```

For example:
```markdown
![Map legend](figures/1.1 "A zoning map showing an 800-meter radius around Metro Station 36...")
```

In [ ]:
# Let's see what CU extracted!

print("🔎 INSPECTING CONTENT UNDERSTANDING OUTPUT\n")
print("="*60)

if not markdown_text:
    print("❌ No markdown text available. Run the analysis or load cached results first.")
else:
    # Show the first part of the markdown
    print("📝 MARKDOWN OUTPUT (first 1500 chars):")
    print("-"*60)
    print(markdown_text[:1500])
    print("-"*60)
    print(f"\n... [{len(markdown_text):,} total characters]")

In [ ]:
# Now let's extract ALL figure descriptions to see what CU generated

print("🖼️ FIGURE DESCRIPTIONS EXTRACTED BY CONTENT UNDERSTANDING\n")

if markdown_text:
    # Pattern to match: ![alt](url "description")
    figure_pattern = re.compile(r'!\[([^\]]*)\]\(([^)]+?)(?:\s+"([^"]+)")?\)', re.DOTALL)
    figures = figure_pattern.findall(markdown_text)
    
    if figures:
        print(f"✅ Found {len(figures)} figure(s) with descriptions:\n")
        
        for i, (alt_text, url, description) in enumerate(figures, 1):
            print(f"{'='*60}")
            print(f"📷 FIGURE {i}")
            print(f"{'='*60}")
            print(f"   URL: {url}")
            print(f"   Alt-text: {alt_text[:80]}..." if len(alt_text) > 80 else f"   Alt-text: {alt_text}")
            
            if description:
                print(f"\n   🎉 AI SEMANTIC DESCRIPTION:")
                print(f"   \"{description[:500]}...\"" if len(description) > 500 else f"   \"{description}\"")
                print("\n   ✨ This description can now be SEARCHED and used by RAG!")
            else:
                print(f"\n   ⚠️ No semantic description (may be using layout-only mode)")
            print()
    else:
        print("⚠️ No figure tags found in markdown output.")
        print("   This may indicate the analyzer ran in layout-only mode.")
else:
    print("❌ No markdown text available.")

---

## Step 5: The Breakthrough - Comparing Module 1 vs Module 3

Let's see the dramatic difference between naive RAG and Content Understanding.

### The Question: "What types of land use surround Station 36?"

| Module 1 (Naive RAG) | Module 3 (Content Understanding) |
|----------------------|----------------------------------|
| Returns: `"מגורים א׳, תעסוקה, מסחר"` | Returns: Full AI description of the zoning map |
| Problem: Just disconnected Hebrew labels | Solution: Explains WHERE each zone is located |
| The MAP was invisible | The MAP is now described semantically |

In [ ]:
# Let's demonstrate the breakthrough with a specific example

print("🎯 THE BREAKTHROUGH: From 'Invisible' to 'Searchable'\n")
print("="*60)

# What naive RAG would have found (simulated)
naive_result = """
מגורים א׳
תעסוקה
מסחר ותעסוקה
מבני ציבור
שטחים פתוחים
"""

print(">>> MODULE 1 (Naive RAG) - What we found:")
print("-"*60)
print(naive_result)
print("-"*60)
print("❌ Problem: Just text labels extracted from the legend.")
print("   We have NO IDEA where these zones are on the map!")
print("   If someone asks 'What is east of the station?' we can't answer.")

print("\n\n>>> MODULE 3 (Content Understanding) - What we now have:")
print("-"*60)

# Search for map description in the CU output
if markdown_text:
    # Look for figure descriptions containing map-related content
    map_keywords = ["map", "zoning", "radius", "מפה", "אזור", "residential", "commercial"]
    
    figure_pattern = re.compile(r'!\[([^\]]*)\]\(([^)]+?)\s+"([^"]+)"\)', re.DOTALL)
    figures = figure_pattern.findall(markdown_text)
    
    map_description = None
    for alt, url, desc in figures:
        desc_lower = desc.lower()
        if any(kw in desc_lower for kw in map_keywords):
            map_description = desc
            break
    
    if map_description:
        print(f"\"{map_description}\"")
        print("-"*60)
        print("✅ SUCCESS! The AI now understands:")
        print("   - What the map shows (zoning around station)")
        print("   - What the colors represent (land use types)")
        print("   - Spatial relationships (what's north, south, east, west)")
        print("\n   🎉 This description is now SEARCHABLE by our RAG pipeline!")
    else:
        print("(Map description not found in cached results)")
        print("\nShowing example of what CU typically generates:")
        print('"A zoning map showing an 800-meter radius around Metro Station 36.')
        print('The map displays different land use zones including residential areas (pink),')
        print('commercial zones (blue), mixed-use areas, and public facilities.')
        print('The station is centrally located with residential neighborhoods')
        print('to the northeast and commercial districts to the southwest."')
else:
    print("(Run the analysis to see actual CU output)")

---

## Step 6: Semantic Chunking with Markdown Headers

Content Understanding outputs **structured markdown** with proper heading hierarchy:

```markdown
# תחנה 36 - שדרות הציונות
## מיקום התחנה
The station is located at...
## שימושי קרקע
Land use in the area includes...
```

We can use these headers for **Semantic Chunking** instead of naive fixed-size chunking:

| Naive Chunking (Module 1) | Semantic Chunking (Module 3) |
|--------------------------|-----------------------------|
| Cut every 500 characters | Cut at section boundaries |
| Breaks mid-sentence | Preserves complete topics |
| Loses context | Keeps related info together |

In [ ]:
# Parse markdown headers for semantic chunking

def chunk_by_headers(text, min_level=1, max_level=2):
    """
    Split markdown text into semantic chunks based on headers.
    
    Args:
        text: Markdown text from Content Understanding
        min_level: Minimum header level (1 = #)
        max_level: Maximum header level (2 = ##)
    
    Returns:
        List of chunks with title, content, and level
    """
    chunks = []
    
    # Pattern to match markdown headers
    header_pattern = re.compile(r'^(#{1,6})\s+(.+)$', re.MULTILINE)
    
    # Find all headers
    headers = [(m.start(), len(m.group(1)), m.group(2).strip()) 
               for m in header_pattern.finditer(text)]
    
    if not headers:
        return [{"title": "Document", "content": text, "level": 0}]
    
    # Create chunks between headers
    for i, (pos, level, title) in enumerate(headers):
        if level < min_level or level > max_level:
            continue
        
        # Find end of chunk
        end_pos = len(text)
        for j in range(i + 1, len(headers)):
            next_pos, next_level, _ = headers[j]
            if next_level <= level:
                end_pos = next_pos
                break
        
        # Extract content
        header_end = text.find('\n', pos)
        if header_end == -1:
            header_end = pos + len(title) + level + 1
        content = text[header_end:end_pos].strip()
        
        if content:
            chunks.append({
                "title": title,
                "content": content,
                "level": level
            })
    
    return chunks

print("📝 SEMANTIC CHUNKING FROM CONTENT UNDERSTANDING OUTPUT\n")

if markdown_text:
    semantic_chunks = chunk_by_headers(markdown_text, min_level=1, max_level=2)
    
    print(f"✅ Created {len(semantic_chunks)} semantic chunks\n")
    
    # Show chunk distribution
    level_counts = {}
    for c in semantic_chunks:
        level_counts[c['level']] = level_counts.get(c['level'], 0) + 1
    
    print("   📊 Chunks by header level:")
    for level, count in sorted(level_counts.items()):
        print(f"      {'#' * level} (level {level}): {count} chunks")
    
    # Show first 3 chunks
    print("\n" + "="*60)
    print("SAMPLE CHUNKS:")
    print("="*60)
    
    for i, chunk in enumerate(semantic_chunks[:3]):
        print(f"\n--- Chunk {i+1}: {chunk['title'][:50]}... ---")
        print(f"    Level: {'#' * chunk['level']}")
        print(f"    Content length: {len(chunk['content'])} chars")
        content_preview = chunk['content'][:200].replace('\n', ' ')
        print(f"    Preview: {content_preview}...")
else:
    print("❌ No markdown text available for chunking.")
    semantic_chunks = []

In [ ]:
# Compare: Naive (fixed-size) vs Semantic chunking

print("🔍 COMPARISON: Naive vs Semantic Chunking\n")
print("="*60)

if markdown_text and len(markdown_text) > 1000:
    # Simulate naive chunking
    naive_chunk_size = 500
    naive_chunk = markdown_text[200:200+naive_chunk_size]
    
    print(">>> NAIVE CHUNKING (500 chars, arbitrary cut):")
    print("-"*60)
    print(f"'{naive_chunk[:100]}...'")
    print(f"... (cut at char 500) ...")
    print(f"'...{naive_chunk[-100:]}'")
    print("-"*60)
    print("❌ Problem: Cuts mid-sentence, mid-paragraph, or mid-figure description!")
    
    print("\n\n>>> SEMANTIC CHUNKING (by section header):")
    print("-"*60)
    
    if semantic_chunks:
        example = semantic_chunks[0]
        print(f"Title: {example['title']}")
        print(f"Content ({len(example['content'])} chars):")
        print(f"'{example['content'][:300]}...'")
        print("-"*60)
        print("✅ Benefit: Entire section stays together as one semantic unit!")
        print("   - No mid-sentence cuts")
        print("   - Figure descriptions stay with their context")
        print("   - Related information isn't scattered across chunks")
else:
    print("(Not enough text for comparison demo)")

---

## 📊 Summary: What Content Understanding Adds

We've now completed the **extraction** phase of our RAG pipeline with three levels of sophistication:

| Module | Technology | Figures | Tables | Structure |
|--------|-----------|---------|--------|----------|
| **Module 1** | Naive (PyPDF) | ❌ Invisible | ❌ Destroyed | ❌ Lost |
| **Module 2** | Document Intelligence | ✅ Bounding boxes | ✅ Rows/columns | ✅ Roles |
| **Module 3** | Content Understanding | ✅ **AI descriptions** | ✅ Rows/columns | ✅ Markdown headers |

### 🎯 Key Takeaways

1. **Content Understanding = Document Intelligence + GPT-4.1-mini**
   - It builds on DI's structure extraction
   - Adds semantic understanding via LLM

2. **Figures are now searchable**
   - No longer just bounding boxes
   - AI-generated descriptions can be indexed and retrieved

3. **Markdown output enables semantic chunking**
   - Headers provide natural chunk boundaries
   - No more arbitrary character cuts

4. **Trade-off: Cost vs Quality**
   - CU costs more (LLM calls for each figure)
   - But the quality improvement is significant for visual documents

---

## ➡️ What's Next?

| Module | What You'll Learn |
|--------|-------------------|
| **Module 4** | Chunking Strategies - Different approaches for different content types |
| **Module 5** | Azure AI Search - Indexing and hybrid search |
| **Module 6** | GraphRAG - Cross-document reasoning |

**Next**: [Module 4 – Chunking Strategies](../module-4-chunking/README.md)